![An image](../workflow.png)

---
# *work flow*
## 1. Configuration & Path Setup

### 1.1 — Defining our data

The AI4Mars dataset is organised into several subdirectories. We define global path constants that will be used throughout the pipeline:

| Variable | Contents |
|:---|:---|
| `IMAGE_`  | Raw EDR camera images (JPG) |
| `MASK__TRAIN` | Training segmentation masks (PNG) |
| `MASK__TEST`  | High-confidence test masks |
| `MASK_ROVER` | Rover presence masks |
| `RANGE_30M` | 30-metre range maps |

In [ ]:
IMAGE_PATH      = "ai4mars_data/ai4mars-dataset-merged-0.1/msl/images/edr/"
MASK_PATH_TRAIN = "ai4mars_data/ai4mars-dataset-merged-0.1/msl/labels/train/"
MASK_PATH_TEST  = "ai4mars_data/ai4mars-dataset-merged-0.1/msl/labels/test/masked-gold-min3-100agree/"
MASK_ROVER      = "ai4mars_data/ai4mars-dataset-merged-0.1/msl/images/mxy/"
RANGE_30M       = "ai4mars_data/ai4mars-dataset-merged-0.1/msl/images/rng-30m/"


---
## 2. Data Preparation

### 2.1 — Building the Image Inventory (`create_df`)

Not all images in the dataset are usable. We apply two filters when building our DataFrame:

1. **Rover visibility filter** — images where the rover body is visible are excluded. The MXY mask encodes rover presence with pixel value `1`. Any image containing such pixels is skipped to avoid training the model on occluded terrain.
2. **Known bad entry** — one specific image (`NLB_432655207EDR_...`) has no corresponding range or rover mask and is excluded manually.

The result is a `pandas` DataFrame with one column — `id` — containing the clean filenames ready for training.

In [ ]:
def create_df(path):
    name = []
    for dirname, _, filenames in os.walk(path):
        for filename in filenames:
            imname = filename.split('.')[0].replace("_merged", "")

            # Skip images where the rover body is visible
            rover_raw = np.array(cv2.imread(MASK_ROVER + imname.replace("EDR", "MXY") + ".png"))
            rover = rover_raw[:, :, 0]
            if (rover == 1).any(): continue

            # Skip known bad entry (missing range/rover masks)
            if imname == "NLB_432655207EDR_F0160148NCAM00394M1": continue

            name.append(imname)

    return pd.DataFrame({'id': name})

### 2.2 — Cache DataFrames to Disk

Building the DataFrame by walking thousands of files is slow. We cache the result as CSV files so subsequent runs load instantly. If a cached version already exists it is read from disk; otherwise it is generated and saved.

In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)

if not os.path.isfile("/kaggle/input/mars-models/df_train.csv"):
    df_train = create_df(MASK_PATH_TRAIN)
    df_train.to_csv("/kaggle/working/df_train.csv", index=False)
else:
    df_train = pd.read_csv("/kaggle/input/mars-models/df_train.csv")

if not os.path.isfile("/kaggle/input/mars-models/df_test.csv"):
    df_test = create_df(MASK_PATH_TEST)
    df_test.to_csv("/kaggle/working/df_test.csv", index=False)
else:
    df_test = pd.read_csv("/kaggle/input/mars-models/df_test.csv")

print(f"Train images : {len(df_train)}")
print(f"Test  images : {len(df_test)}")

### 2.3 — Label Remapping & Visualisation

The raw mask PNG files use pixel value `255` for **NULL** (unlabelled) pixels. This value is incompatible with standard cross-entropy loss (which expects class indices `0 … N-1`), so we remap:

```
255  →  4   (NULL class)
```

After remapping every pixel falls into the range `[0, 4]`.

We then visualise a sample that contains all four terrain classes at once — plotting each binary class mask separately and overlaying the full label on the raw grayscale image.

In [ ]:
import matplotlib.pyplot as plt

# Load a sample image and its mask
sample_id = df_train['id'].values[39]
gt_raw = np.array(cv2.imread(MASK_PATH_TRAIN + sample_id + ".png"))
img_raw = np.array(cv2.imread(IMAGE_PATH + sample_id + ".JPG"))

ground_truth = gt_raw[:, :, 0].astype(float)
img = img_raw[:, :, 0]

# Remap NULL (255) → class 4
ground_truth[ground_truth == 255] = 4

# Visualise each class as a binary mask
class_names = ['Soil (0)', 'Bedrock (1)', 'Sand (2)', 'Big Rock (3)', 'NULL (4)']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(np.where(ground_truth == i, 1, 0), cmap='hot')
    ax.set_title(class_names[i])
    ax.axis('off')
plt.suptitle('Per-class Binary Masks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4 — Train / Validation / Test Split

We split the data into three non-overlapping sets using `sklearn`'s `train_test_split`:

| Split | Source | Proportion | Purpose |
|:---:|:---|:---:|:---|
| **Train** | `df_train` (80%) | ~80% of trainval | Model weight updates |
| **Validation** | `df_train` (20%) | ~20% of trainval | Hyperparameter tuning, early stopping |
| **Test** | `df_test` | Separate held-out set | Final unbiased evaluation |

The test set uses the **gold-standard** labels (`min3-100agree`) — images that required at least 3 annotators to reach 100% agreement, making them the most reliable ground truth available.

In [ ]:
from sklearn.model_selection import train_test_split

X_trainval = df_train['id'].values
X_test     = df_test['id'].values
X_train, X_val = train_test_split(X_trainval, test_size=0.20, random_state=42)

print(f"{'Split':<12} {'Size':>6}")
print("-" * 20)
print(f"{'Train':<12} {len(X_train):>6}")
print(f"{'Validation':<12} {len(X_val):>6}")
print(f"{'Test':<12} {len(X_test):>6}")

---
## 3. Dataset & DataLoader

### 3.1 — `MarsDataset` Class

We implement a custom PyTorch `Dataset` that handles the specifics of the AI4Mars data:

- **Images** are loaded in BGR by OpenCV and converted to **RGB** for compatibility with ImageNet-pretrained backbones
- **Masks** are loaded as grayscale and the NULL remap (`255 → 4`) is applied at load time
- An optional **Albumentations transform** is applied jointly to both image and mask, ensuring spatial augmentations are consistent

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

class MarsDataset(Dataset):
    """Loads (image, mask) pairs for Mars terrain segmentation."""

    def __init__(self, img_ids, image_path, mask_path, transform=None):
        self.img_ids    = img_ids
        self.image_path = image_path
        self.mask_path  = mask_path
        self.transform  = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]

        # Load & convert image to RGB
        img = cv2.imread(self.image_path + img_id + ".JPG")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load mask and remap NULL pixels
        mask = cv2.imread(self.mask_path + img_id + ".png", cv2.IMREAD_GRAYSCALE).astype(np.int64)
        mask[mask == 255] = 4

        if self.transform:
            aug = self.transform(image=img, mask=mask.astype(np.uint8))
            img, mask = aug['image'], aug['mask'].long()
        else:
            img  = T.ToTensor()(img)
            mask = torch.tensor(mask, dtype=torch.long)

        return img, mask

### 3.2 — Augmentation Pipelines

Data augmentation helps the model generalise to unseen images. We use **Albumentations** for its speed and native mask-aware transforms:

| Transform | Train | Val/Test | Purpose |
|:---|:---:|:---:|:---|
| `Resize(512, 512)` | ✅ | ✅ | Uniform input size |
| `HorizontalFlip(p=0.5)` | ✅ | ❌ | Left-right symmetry |
| `RandomBrightnessContrast(p=0.3)` | ✅ | ❌ | Lighting variation |
| `Normalize` (ImageNet stats) | ✅ | ✅ | Match pretrained backbone expectations |
| `ToTensorV2` | ✅ | ✅ | Convert to PyTorch tensor |

In [ ]:
import albumentations as A

IMG_SIZE = 512

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    A.pytorch.ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    A.pytorch.ToTensorV2(),
])

# Instantiate datasets and loaders
BATCH_SIZE = 4
train_ds = MarsDataset(X_train, IMAGE_PATH, MASK_PATH_TRAIN, transform=train_transform)
val_ds   = MarsDataset(X_val,   IMAGE_PATH, MASK_PATH_TRAIN, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


---
## 4. Model Setup

### 4.1 — Why DeepLabV3+?

**DeepLabV3+** is a state-of-the-art semantic segmentation architecture that uses:

- **Atrous (dilated) convolutions** — capture multi-scale context without losing resolution
- **ASPP (Atrous Spatial Pyramid Pooling)** — processes features at multiple dilation rates
- **ResNet-101 backbone** — deep feature extractor pretrained on ImageNet
- **Auxiliary loss head** — an extra intermediate supervision signal that improves gradient flow during training

We use the pretrained weights and replace only the **final classification layer** to output our 5 classes.

### 4.2 — Class-Weighted Loss

The terrain classes are **imbalanced** — soil and bedrock dominate, while big rocks are rare. We address this with a weighted `CrossEntropyLoss`:

| Class | Weight | Reason |
|:---:|:---:|:---|
| Soil | 1.0 | Common, baseline |
| Bedrock | 1.0 | Common, baseline |
| Sand | 1.0 | Moderate frequency |
| **Big Rock** | **6.0** | Rare  |
| NULL | 0.5 | Not a real class — downweighted |

In [ ]:

EPOCHS = 22

# Load pretrained DeepLabV3+ with ResNet-101 backbone
model = seg_models.deeplabv3_resnet101(weights=seg_models.DeepLabV3_ResNet101_Weights.DEFAULT)

# Replace the classifier heads for our 5-class problem
model.classifier[4]     = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
model.aux_classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
model = model.to(DEVICE)

# Weighted loss: upweight big rocks (rare class), downweight NULL
class_weights = torch.tensor([1.0, 1.0, 1.0, 6.0, 0.5], device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)


### 4.3 — Checkpoint Auto-Resume

Training 22 epochs on this dataset takes hours. To avoid losing progress if the Colab session crashes, we save a **full checkpoint** to Google Drive after every epoch. On startup the code automatically finds the latest checkpoint and resumes from the correct epoch.

If no per-epoch checkpoint exists, it falls back to loading an earlier base model checkpoint, and if that is also absent it starts from ImageNet weights.

In [ ]:
START_EPOCH = 1

# Search for the latest checkpoint saved to Drive
latest_ckpt = None
for ep in range(EPOCHS, 0, -1):
    path = f"{SAVE_DIR}checkpoint_epoch{ep:02d}.pth"
    if os.path.exists(path):
        latest_ckpt = path
        break
#====<>=======
if latest_ckpt:
    ckpt = torch.load(latest_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    history     = ckpt['history']
    START_EPOCH = ckpt['epoch'] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} → continuing from epoch {START_EPOCH}")
else:
    print("⚠️  No checkpoint found — starting from ImageNet weights")

---
## 5. Training Loop

### 5.1 — Evaluation Metrics

We track two metrics during training:

**Pixel Accuracy** — the fraction of pixels correctly classified across all classes:
$$\text{Pixel Accuracy} = \frac{\text{Correctly classified pixels}}{\text{Total pixels}}$$


![An image](https://cdn.prod.website-files.com/614c82ed388d53640613982e/64b90a7ad0f40f9dd65af70e_iou%20formula.webp)

### 5.2 — Per-Epoch Procedure

Each epoch consists of:
1. **Train phase** — forward pass, compute weighted cross-entropy + 0.4 × auxiliary loss, backpropagate, update weights
2. **Validation phase** — forward pass only (no gradients), compute val loss, pixel accuracy, and mIoU
3. **LR Scheduling** — `ReduceLROnPlateau` halves the learning rate if mIoU doesn't improve for 3 epochs
4. **Checkpoint save** — full model + optimizer + scheduler state saved to Google Drive

In [ ]:
from tqdm.notebook import tqdm

def pixel_accuracy(preds, masks):
    return (preds == masks).sum().item() / masks.numel()

def mean_iou(preds, masks, num_classes=NUM_CLASSES):
    ious = []
    for cls in range(num_classes - 1):   # exclude NULL (class 4)
        inter = ((preds == cls) & (masks == cls)).sum().item()
        union = ((preds == cls) | (masks == cls)).sum().item()
        if union > 0:
            ious.append(inter / union)
    return np.mean(ious) if ious else 0.0


torch.cuda.empty_cache()

for epoch in range(START_EPOCH, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs['out'], masks) + 0.4 * criterion(outputs['aux'], masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_acc, val_miou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]", leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            out      = model(imgs)['out']
            val_loss += criterion(out, masks).item()
            preds     = out.argmax(dim=1)
            val_acc  += pixel_accuracy(preds, masks)
            val_miou += mean_iou(preds.cpu().numpy(), masks.cpu().numpy())

    val_loss  /= len(val_loader)
    val_acc   /= len(val_loader)
    val_miou  /= len(val_loader)
    scheduler.step(val_miou)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_miou"].append(val_miou)

    print(f"Epoch {epoch:02d} | Loss {train_loss:.4f} → {val_loss:.4f} | Acc {val_acc:.4f} | mIoU {val_miou:.4f}")

    # ── Save checkpoint ────────────────────────────────────────────────────────
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history': history},
               f"{SAVE_DIR}checkpoint_epoch{epoch:02d}.pth")

---
## 6. Evaluation & Visualisation

### 6.1 — Training Curves

After training completes we plot three learning curves to inspect model behaviour:

- **Loss** (train vs val) — should both decrease; a large gap indicates overfitting
- **Val Pixel Accuracy** — how many pixels are correctly labelled on the validation set
- **Val Mean IoU** — the primary metric; higher is better, computed over 4 terrain classes

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].set_title('Loss (Train vs Val)'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['val_acc'], color='green')
axes[1].set_title('Val Pixel Accuracy'); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_range, history['val_miou'], color='orange')
axes[2].set_title('Val Mean IoU (4 classes)'); axes[2].set_xlabel('Epoch')

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 — Cherry-Picked Visualisation

To qualitatively assess the model we find a validation sample that contains all 4 terrain classes and display it as a three-panel figure:

- **Left** — raw input image (after de-normalisation)
- **Centre** — ground-truth segmentation mask with class colours
- **Right** — DeepLabV3+ predicted segmentation mask

We also report **per-class IoU** for that specific image so we can see which classes the model handles well and which it struggles with.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

CLASS_COLORS = [
    (0.76, 0.60, 0.42),  # soil      – sandy brown
    (0.50, 0.35, 0.20),  # bedrock   – dark brown
    (0.94, 0.85, 0.60),  # sand      – light yellow
    (0.30, 0.30, 0.30),  # bigrock   – dark grey
    (0.20, 0.20, 0.80),  # null      – blue
]
CLASS_NAMES = ['Soil', 'Bedrock', 'Sand', 'Big Rock', 'NULL']

def colorize(mask_2d):
    rgb = np.zeros((*mask_2d.shape, 3))
    for cls, color in enumerate(CLASS_COLORS):
        rgb[mask_2d == cls] = color
    return rgb

def denormalize(tensor):
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).cpu().numpy()
    return np.clip(std * img + mean, 0, 1)

# Find a sample with all 4 terrain classes
model.eval()
for imgs, masks in val_loader:
    for i in range(imgs.shape[0]):
        m = masks[i].numpy()
        if all((m == c).any() for c in range(4)):
            img_t = imgs[i].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pred_np = model(img_t)['out'].argmax(dim=1).squeeze(0).cpu().numpy()
            gt_np = m
            break
    else:
        continue
    break

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Cherry-picked Validation Sample', fontsize=16, fontweight='bold')
axes[0].imshow(denormalize(img_t.squeeze(0)));  axes[0].set_title('Raw Image');             axes[0].axis('off')
axes[1].imshow(colorize(gt_np));                axes[1].set_title('Ground Truth');           axes[1].axis('off')
axes[2].imshow(colorize(pred_np));              axes[2].set_title('DeepLabV3+ Prediction');  axes[2].axis('off')

patches = [mpatches.Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=12, bbox_to_anchor=(0.5, -0.04), frameon=False)
plt.tight_layout()
plt.show()